# Chunk Evaluation Pipeline
Build chunk variants, run benchmark, evaluate, and compare results.

In [ ]:
from pathlib import Path
import subprocess, shutil

def find_repo_root(start: Path) -> Path:
    p = start.resolve()
    for cur in [p] + list(p.parents):
        if (cur / 'config.py').exists() and (cur / 'data').exists():
            return cur
    raise RuntimeError('Cannot find repo root')

ROOT = find_repo_root(Path.cwd())

def display_path(path: Path | str) -> str:
    p = Path(path).resolve() if not isinstance(path, Path) else path.resolve()
    root = ROOT.resolve()
    try:
        rel = p.relative_to(root)
        return ROOT.name if str(rel) == '.' else f"{ROOT.name}/{rel.as_posix()}"
    except Exception:
        return str(path)

print('ROOT =', display_path(ROOT))

def run(cmd):
    print('\n$ ' + ' '.join(cmd))
    subprocess.run(cmd, cwd=ROOT, check=True)

GT = ROOT / 'data' / 'qa_pairs' / 'eval_ground_truth.jsonl'
if not GT.exists():
    raise FileNotFoundError(f'Ground truth not found: {display_path(GT)}')

ROOT = /Users/ben/Library/CloudStorage/GoogleDrive-rs.runsheng@gmail.com/My Drive/Colab Notebooks/CS614 GenAI/qa_agent_legal_tax


In [9]:
# 1) Build two chunk variants
run(['python', 'src/eval/chunking.py'])


$ python src/eval/chunking.py
Source directory: /Users/ben/Library/CloudStorage/GoogleDrive-rs.runsheng@gmail.com/My Drive/Colab Notebooks/CS614 GenAI/qa_agent_legal_tax/data/acts_md
Fixed chunk output directory: /Users/ben/Library/CloudStorage/GoogleDrive-rs.runsheng@gmail.com/My Drive/Colab Notebooks/CS614 GenAI/qa_agent_legal_tax/data/tmp/acts_chunked_fixed
Structure-aware chunk output directory: /Users/ben/Library/CloudStorage/GoogleDrive-rs.runsheng@gmail.com/My Drive/Colab Notebooks/CS614 GenAI/qa_agent_legal_tax/data/tmp/acts_chunked_struct
[OK] strategy=fixed, files=31, out=/Users/ben/Library/CloudStorage/GoogleDrive-rs.runsheng@gmail.com/My Drive/Colab Notebooks/CS614 GenAI/qa_agent_legal_tax/data/tmp/acts_chunked_fixed
[OK] strategy=struct, files=31, out=/Users/ben/Library/CloudStorage/GoogleDrive-rs.runsheng@gmail.com/My Drive/Colab Notebooks/CS614 GenAI/qa_agent_legal_tax/data/tmp/acts_chunked_struct


In [11]:
# 2) Swap active chunk dir and run eval for each strategy
active = ROOT / 'data' / 'acts_chunked'
fixed = ROOT / 'data' / 'tmp' / 'acts_chunked_fixed'
struct = ROOT / 'data' / 'tmp' / 'acts_chunked_struct'
backup = ROOT / 'data' / 'tmp' / 'acts_chunked__backup_for_chunk_eval'

def activate_variant(src_dir: Path):
    if backup.exists():
        shutil.rmtree(backup)
    if active.exists():
        active.rename(backup)
    shutil.copytree(src_dir, active)

def restore_active():
    if active.exists():
        shutil.rmtree(active)
    if backup.exists():
        backup.rename(active)

try:
    # fixed
    activate_variant(fixed)
    run(['python', 'scripts/eval/run_eval_benchmark.py', '--gt', str(GT), '--out', 'outputs/preds_fixed.jsonl', '--enable-rerank', 'false'])
    run(['python', 'scripts/eval/evaluate_predictions.py', '--gt', str(GT), '--pred', 'outputs/preds_fixed.jsonl', '--k', '5', '--out', 'outputs/eval_fixed.json'])

    # struct
    activate_variant(struct)
    run(['python', 'scripts/eval/run_eval_benchmark.py', '--gt', str(GT), '--out', 'outputs/preds_struct.jsonl', '--enable-rerank', 'false'])
    run(['python', 'scripts/eval/evaluate_predictions.py', '--gt', str(GT), '--pred', 'outputs/preds_struct.jsonl', '--k', '5', '--out', 'outputs/eval_struct.json'])
finally:
    restore_active()


$ python scripts/eval/run_eval_benchmark.py --gt /Users/ben/Library/CloudStorage/GoogleDrive-rs.runsheng@gmail.com/My Drive/Colab Notebooks/CS614 GenAI/qa_agent_legal_tax/data/qa_pairs/eval_ground_truth.jsonl --out outputs/preds_fixed.jsonl --enable-rerank false
[debug] QAAgent.__init__ signature: (self, retriever, llm_service, validator, config: Dict)
[debug] cfg_dict keys (48): ['ACTS_CHUNKED_DIR', 'ACTS_CSV', 'ACTS_HTML_DIR', 'ACTS_MD_DIR', 'BM25_B', 'BM25_K1', 'CHUNK_OVERLAP', 'CHUNK_SIZE', 'COHERE_API_KEY', 'COHERE_RERANK_MODEL', 'DATA_DIR', 'DEBUG', 'DOCS_DIR', 'EMBEDDING_DIMENSION', 'EMBEDDING_MODEL', 'EMBEDDING_PROVIDER', 'ENABLE_KG', 'ENABLE_RERANK', 'FINANCIAL_TEMPLATES_DIR', 'GEMINI_EMBEDDING_MODEL', 'GEMINI_LLM_MODEL', 'GOOGLE_API_KEY', 'KG_BOOST_WEIGHT', 'KG_MAX_EXPANSION', 'LLM_MAX_TOKENS', 'LLM_MODEL', 'LLM_PROVIDER', 'LLM_TEMPERATURE', 'LOGS_DIR', 'LOG_FILE', 'LOG_FORMAT', 'LOG_LEVEL', 'MIN_CONFIDENCE_SCORE', 'NOTEBOOKS_DIR', 'OPENAI_API_KEY', 'OUTPUTS_DIR', 'PROJECT_R

Error in batch embedding: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}
Vector search failed, fallback to BM25/keyword: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}
Error in batch embedding: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}
Vector search failed, fallback to BM25/keyword: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/text-embedding-00

CalledProcessError: Command '['python', 'scripts/eval/run_eval_benchmark.py', '--gt', '/Users/ben/Library/CloudStorage/GoogleDrive-rs.runsheng@gmail.com/My Drive/Colab Notebooks/CS614 GenAI/qa_agent_legal_tax/data/qa_pairs/eval_ground_truth.jsonl', '--out', 'outputs/preds_fixed.jsonl', '--enable-rerank', 'false']' returned non-zero exit status 1.

In [ ]:
# 3) Compare and print markdown result
run([
    'python', 'scripts/eval/chunk_eval/eval_chunking_compare.py',
    '--fixed', 'outputs/eval_fixed.json',
    '--struct', 'outputs/eval_struct.json',
    '--fixed-name', 'Fixed-size chunking',
    '--struct-name', 'Structure-aware chunking',
    '--out-md', 'outputs/eval_chunking_comparison.md',
    '--out-json', 'outputs/eval_chunking_comparison.json'
])

print('\n===== outputs/eval_chunking_comparison.md =====')
print((ROOT / 'outputs' / 'eval_chunking_comparison.md').read_text(encoding='utf-8'))